# Machine Configuration Analysis

Explore and analyze machine configurations, monitoring parameters, and recipe parameters.
Load configurations from the database and generate insights about parameter coverage.

## 1. Import Required Libraries

In [1]:
import os
import sys
import json
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime

# Add project root to path
project_root = os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath('cable_maintenance_ai'))))
sys.path.insert(0, project_root)

print(f"Project root: {project_root}")

Project root: c:\Users\stagiaire5\Desktop\coficab_agent-main


## 2. Database Connection Setup

In [2]:
# Load environment variables
load_dotenv()

# Database configuration
DB_HOST = os.getenv('DB_HOST')
DB_PORT = os.getenv('DB_PORT', '1433')  # SQL Server default port
DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD', '')
DB_NAME = os.getenv('DB_NAME')

# Create connection string for SQL Server
password_encoded = quote_plus(DB_PASSWORD)
# SQL Server uses comma between host and port, not colon
db_url = f"mssql+pyodbc://{DB_USER}:{password_encoded}@{DB_HOST},{DB_PORT}/{DB_NAME}?driver=ODBC+Driver+18+for+SQL+Server&TrustServerCertificate=yes"

# Create engine
engine = create_engine(db_url, echo=False, pool_pre_ping=True)

# Test connection
try:
    with engine.connect() as conn:
        result = conn.execute(text("SELECT 1"))
        print("✅ Database connection successful!")
except Exception as e:
    print(f"❌ Database connection failed: {e}")

✅ Database connection successful!


## 2.5 Initialize Database Tables for Report Storage

In [3]:
def initialize_report_tables():
    """Create report storage tables in model_schema if they don't exist."""
    try:
        with engine.connect() as conn:
            # Configuration Reports Table - stores CSV exports as JSON
            conn.execute(text("""
                CREATE TABLE IF NOT EXISTS `model_schema`.`configuration_reports` (
                    `ReportId` INT AUTO_INCREMENT PRIMARY KEY,
                    `ReportName` VARCHAR(255) NOT NULL,
                    `ReportType` VARCHAR(100) NOT NULL COMMENT 'e.g., configuration_export',
                    `ReportData` LONGTEXT NOT NULL COMMENT 'JSON array of report rows',
                    `Summary` JSON COMMENT 'Summary statistics as JSON',
                    `GeneratedAt` TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                    `ProcessedRows` INT DEFAULT 0,
                    INDEX `idx_report_type` (`ReportType`),
                    INDEX `idx_generated` (`GeneratedAt`)
                ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci
            """))
            
            # Coverage Analysis Results Table
            conn.execute(text("""
                CREATE TABLE IF NOT EXISTS `model_schema`.`coverage_analysis_results` (
                    `AnalysisId` INT AUTO_INCREMENT PRIMARY KEY,
                    `ConfigurationName` VARCHAR(255) NOT NULL,
                    `MachineCode` VARCHAR(100) NOT NULL,
                    `AvailableTags` INT,
                    `MonitoredTags` INT,
                    `RecipeTags` INT,
                    `CoveragePct` DECIMAL(5, 2),
                    `RecipePct` DECIMAL(5, 2),
                    `AnalyzedAt` TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                    INDEX `idx_machine` (`MachineCode`),
                    INDEX `idx_config` (`ConfigurationName`),
                    INDEX `idx_analyzed` (`AnalyzedAt`)
                ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci
            """))
            
            # Parameter Statistics Table
            conn.execute(text("""
                CREATE TABLE IF NOT EXISTS `model_schema`.`parameter_statistics` (
                    `StatId` INT AUTO_INCREMENT PRIMARY KEY,
                    `Parameter` VARCHAR(255) NOT NULL,
                    `MonitoringCount` INT DEFAULT 0,
                    `RecipeCount` INT DEFAULT 0,
                    `AnalyzedAt` TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                    UNIQUE KEY `unique_param_analysis` (`Parameter`, `AnalyzedAt`),
                    INDEX `idx_parameter` (`Parameter`),
                    INDEX `idx_analyzed` (`AnalyzedAt`)
                ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci
            """))
            
            conn.commit()
            print("✅ Report storage tables initialized successfully")
    except Exception as e:
        print(f"Note: Tables already exist or initialization skipped: {str(e)}")

# Initialize tables
initialize_report_tables()

Note: Tables already exist or initialization skipped: (pyodbc.ProgrammingError) ('42000', "[42000] [Microsoft][ODBC Driver 18 for SQL Server][SQL Server]Syntaxe incorrecte vers le mot clé 'IF'. (156) (SQLExecDirectW); [42000] [Microsoft][ODBC Driver 18 for SQL Server][SQL Server]Syntaxe incorrecte vers '`'. (102)")
[SQL: 
                CREATE TABLE IF NOT EXISTS `model_schema`.`configuration_reports` (
                    `ReportId` INT AUTO_INCREMENT PRIMARY KEY,
                    `ReportName` VARCHAR(255) NOT NULL,
                    `ReportType` VARCHAR(100) NOT NULL COMMENT 'e.g., configuration_export',
                    `ReportData` LONGTEXT NOT NULL COMMENT 'JSON array of report rows',
                    `Summary` JSON COMMENT 'Summary statistics as JSON',
                    `GeneratedAt` TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                    `ProcessedRows` INT DEFAULT 0,
                    INDEX `idx_report_type` (`ReportType`),
                    INDEX `idx_genera

## 3. Load Machine Configurations from Database

In [4]:
def load_configurations():
    """Load all machine configurations from database."""
    try:
        query = """
        SELECT 
            ConfigurationId,
            ConfigurationName,
            MachineCode,
            MonitoringParameters,
            RecipeParameters,
            Description,
            IsActive,
            CreatedAt,
            UpdatedAt
        FROM `model_schema`.`machine_configuration`
        ORDER BY ConfigurationName
        """
        df = pd.read_sql(text(query), engine)
        
        # Parse JSON columns
        df['MonitoringParameters'] = df['MonitoringParameters'].apply(
            lambda x: json.loads(x) if isinstance(x, str) else (x if isinstance(x, list) else [])
        )
        df['RecipeParameters'] = df['RecipeParameters'].apply(
            lambda x: json.loads(x) if isinstance(x, str) else (x if isinstance(x, list) else [])
        )
        
        return df
    except Exception as e:
        print(f"Error loading configurations: {e}")
        return pd.DataFrame()

configs_df = load_configurations()
print(f"Loaded {len(configs_df)} configurations")
configs_df.head()

Error loading configurations: Execution failed on sql '
        SELECT 
            ConfigurationId,
            ConfigurationName,
            MachineCode,
            MonitoringParameters,
            RecipeParameters,
            Description,
            IsActive,
            CreatedAt,
            UpdatedAt
        FROM `model_schema`.`machine_configuration`
        ORDER BY ConfigurationName
        ': (pyodbc.ProgrammingError) ('42000', "[42000] [Microsoft][ODBC Driver 18 for SQL Server][SQL Server]Syntaxe incorrecte vers '`'. (102) (SQLExecDirectW)")
[SQL: 
        SELECT 
            ConfigurationId,
            ConfigurationName,
            MachineCode,
            MonitoringParameters,
            RecipeParameters,
            Description,
            IsActive,
            CreatedAt,
            UpdatedAt
        FROM `model_schema`.`machine_configuration`
        ORDER BY ConfigurationName
        ]
(Background on this error at: https://sqlalche.me/e/20/f405)
Loaded 0 confi

""


## 4. Configuration Summary Statistics

In [5]:
if not configs_df.empty:
    print("\n=== Configuration Summary ===")
    print(f"Total Configurations: {len(configs_df)}")
    print(f"Active Configurations: {len(configs_df[configs_df['IsActive'] == True])}")
    print(f"Unique Machines: {configs_df['MachineCode'].nunique()}")
    print(f"\nMachines configured:")
    machine_counts = configs_df['MachineCode'].value_counts()
    print(machine_counts)
    
    print(f"\n=== Parameter Statistics ===")
    configs_df['n_monitoring_params'] = configs_df['MonitoringParameters'].apply(len)
    configs_df['n_recipe_params'] = configs_df['RecipeParameters'].apply(len)
    
    print(f"Average monitoring parameters per config: {configs_df['n_monitoring_params'].mean():.2f}")
    print(f"Average recipe parameters per config: {configs_df['n_recipe_params'].mean():.2f}")
    print(f"Max monitoring parameters: {configs_df['n_monitoring_params'].max()}")
    print(f"Min monitoring parameters: {configs_df['n_monitoring_params'].min()}")
else:
    print("No configurations found in database.")

No configurations found in database.


## 5. Load Machine Tags for Analysis

In [ ]:
def load_machine_tags():
    """Load all distinct machine tags from MachineTagValue."""
    try:
        query = """
        SELECT DISTINCT
            MachineCode,
            OpcNodeId,
            MIN(Value) as min_val,
            MAX(Value) as max_val,
            AVG(Value) as mean_val,
            COUNT(*) as sample_count
        FROM MachineTagValue
        GROUP BY MachineCode, OpcNodeId
        ORDER BY MachineCode, OpcNodeId
        """
        df = pd.read_sql(text(query), engine)
        return df
    except Exception as e:
        print(f"Error loading machine tags: {e}")
        return pd.DataFrame()

tags_df = load_machine_tags()
print(f"Loaded {len(tags_df)} machine tag combinations")
print(f"\nUnique machines in MachineTagValue: {tags_df['MachineCode'].nunique()}")
if not tags_df.empty:
    print(f"Machine tag distribution:")
    print(tags_df['MachineCode'].value_counts())

## 6. Configuration Coverage Analysis

In [ ]:
# Analyze coverage: which parameters are monitored vs available
if not configs_df.empty and not tags_df.empty:
    coverage_analysis = []
    
    for _, config in configs_df.iterrows():
        machine = config['MachineCode']
        available_tags = tags_df[tags_df['MachineCode'] == machine]['OpcNodeId'].tolist()
        monitored = config['MonitoringParameters']
        recipes = config['RecipeParameters']
        
        coverage_pct = (len(monitored) / len(available_tags) * 100) if available_tags else 0
        recipe_pct = (len(recipes) / len(monitored) * 100) if monitored else 0
        
        coverage_analysis.append({
            'ConfigurationName': config['ConfigurationName'],
            'MachineCode': machine,
            'AvailableTags': len(available_tags),
            'MonitoredTags': len(monitored),
            'RecipeTags': len(recipes),
            'CoveragePct': coverage_pct,
            'RecipePct': recipe_pct
        })
    
    coverage_df = pd.DataFrame(coverage_analysis)
    
    # Store coverage analysis results in database
    try:
        with engine.connect() as conn:
            # Clear previous analysis from this run
            conn.execute(text("DELETE FROM `model_schema`.`coverage_analysis_results` WHERE DATE(AnalyzedAt) = CURDATE()"))
            
            # Insert new results
            for _, row in coverage_df.iterrows():
                conn.execute(text("""
                    INSERT INTO `model_schema`.`coverage_analysis_results`
                    (ConfigurationName, MachineCode, AvailableTags, MonitoredTags, RecipeTags, CoveragePct, RecipePct)
                    VALUES (:config_name, :machine, :available, :monitored, :recipe, :coverage, :recipe_pct)
                """), {
                    'config_name': row['ConfigurationName'],
                    'machine': row['MachineCode'],
                    'available': int(row['AvailableTags']),
                    'monitored': int(row['MonitoredTags']),
                    'recipe': int(row['RecipeTags']),
                    'coverage': float(row['CoveragePct']),
                    'recipe_pct': float(row['RecipePct'])
                })
            conn.commit()
        print("✅ Coverage analysis results stored in model_schema.coverage_analysis_results")
    except Exception as e:
        print(f"⚠️  Note: Could not store coverage results: {str(e)}")
    
    print("\n=== Configuration Coverage Analysis ===")
    print(coverage_df.to_string(index=False))
else:
    print("No configurations or tags found for coverage analysis.")

## 7. Visualization: Parameters per Configuration

In [ ]:
if not configs_df.empty:
    fig = px.bar(
        configs_df,
        x='ConfigurationName',
        y=['n_monitoring_params', 'n_recipe_params'],
        title='Monitoring vs Recipe Parameters per Configuration',
        labels={
            'ConfigurationName': 'Configuration',
            'value': 'Parameter Count',
            'variable': 'Type'
        },
        barmode='group'
    )
    fig.update_xaxes(tickangle=-45)
    fig.show()
else:
    print("No configurations to visualize.")

## 8. Detailed Configuration Breakdown

In [ ]:
def print_configuration_details(config_id=None):
    """Print detailed information for a specific configuration or all configurations."""
    df = configs_df if config_id is None else configs_df[configs_df['ConfigurationId'] == config_id]
    
    for _, config in df.iterrows():
        print(f"\n{'='*80}")
        print(f"Configuration ID: {config['ConfigurationId']}")
        print(f"Name: {config['ConfigurationName']}")
        print(f"Machine: {config['MachineCode']}")
        print(f"Status: {'🟢 Active' if config['IsActive'] else '🔴 Inactive'}")
        print(f"Created: {config['CreatedAt']}")
        print(f"Updated: {config['UpdatedAt']}")
        
        if config['Description']:
            print(f"Description: {config['Description']}")
        
        monitoring = config['MonitoringParameters']
        recipes = config['RecipeParameters']
        
        print(f"\nMonitoring Parameters ({len(monitoring)}):")
        for param in monitoring:
            is_recipe = "🔷 RECIPE" if param in recipes else "📊 MONITOR"
            print(f"  • {param} {is_recipe}")
        
        print(f"\nRecipe Parameters ({len(recipes)}):")
        for param in recipes:
            print(f"  • {param}")

# Print all configurations
if not configs_df.empty:
    print_configuration_details()
else:
    print("No configurations found in database.")

## 9. Parameter Usage Statistics

In [ ]:
# Analyze which parameters are most commonly used across configurations
if not configs_df.empty:
    param_usage = {}
    recipe_usage = {}
    
    for _, config in configs_df.iterrows():
        for param in config['MonitoringParameters']:
            param_usage[param] = param_usage.get(param, 0) + 1
            if param in config['RecipeParameters']:
                recipe_usage[param] = recipe_usage.get(param, 0) + 1
    
    # Create summary dataframe
    param_summary = pd.DataFrame([
        {
            'Parameter': param,
            'MonitoringCount': param_usage.get(param, 0),
            'RecipeCount': recipe_usage.get(param, 0)
        }
        for param in sorted(set(list(param_usage.keys()) + list(recipe_usage.keys())))
    ])
    
    param_summary = param_summary.sort_values('MonitoringCount', ascending=False)
    
    # Store parameter statistics in database
    try:
        with engine.connect() as conn:
            # Clear previous analysis from this run
            conn.execute(text("DELETE FROM `model_schema`.`parameter_statistics` WHERE DATE(AnalyzedAt) = CURDATE()"))
            
            # Insert new statistics
            for _, row in param_summary.iterrows():
                conn.execute(text("""
                    INSERT INTO `model_schema`.`parameter_statistics`
                    (Parameter, MonitoringCount, RecipeCount)
                    VALUES (:param, :monitoring, :recipe)
                """), {
                    'param': row['Parameter'],
                    'monitoring': int(row['MonitoringCount']),
                    'recipe': int(row['RecipeCount'])
                })
            conn.commit()
        print("✅ Parameter statistics stored in model_schema.parameter_statistics")
    except Exception as e:
        print(f"⚠️  Note: Could not store parameter statistics: {str(e)}")
    
    print("\n=== Parameter Usage Statistics ===")
    print(param_summary.to_string(index=False))
    print(f"\nTotal unique parameters monitored: {len(param_summary)}")
    print(f"Total unique recipe parameters: {len(param_summary[param_summary['RecipeCount'] > 0])}")
else:
    print("No configurations found.")

## 10. Export Configuration Report

## 11. Retrieve Stored Reports and Analysis from Database

In [ ]:
def get_latest_configuration_report():
    """Retrieve the most recent configuration report from database."""
    try:
        with engine.connect() as conn:
            df = pd.read_sql(text("""
                SELECT ReportId, ReportName, ReportType, ProcessedRows, GeneratedAt, Summary
                FROM `model_schema`.`configuration_reports`
                WHERE ReportType = 'configuration_export'
                ORDER BY GeneratedAt DESC
                LIMIT 1
            """), conn)
        
        if df.empty:
            print("No configuration reports found in database.")
            return None
        
        report = df.iloc[0].to_dict()
        summary = json.loads(report['Summary']) if isinstance(report['Summary'], str) else report['Summary']
        
        print(f"✅ Latest Report: {report['ReportName']}")
        print(f"   Generated: {report['GeneratedAt']}")
        print(f"   Rows: {report['ProcessedRows']}")
        print(f"   Summary: {summary}")
        
        return report
    except Exception as e:
        print(f"Error retrieving report: {str(e)}")
        return None

def get_latest_coverage_analysis():
    """Retrieve the most recent coverage analysis from database."""
    try:
        with engine.connect() as conn:
            df = pd.read_sql(text("""
                SELECT ConfigurationName, MachineCode, AvailableTags, MonitoredTags, 
                       RecipeTags, CoveragePct, RecipePct, AnalyzedAt
                FROM `model_schema`.`coverage_analysis_results`
                ORDER BY AnalyzedAt DESC
            """), conn)
        
        if df.empty:
            print("No coverage analysis results found in database.")
            return None
        
        print(f"✅ Coverage Analysis Results ({len(df)} rows)")
        print(df.to_string(index=False))
        return df
    except Exception as e:
        print(f"Error retrieving coverage analysis: {str(e)}")
        return None

def get_latest_parameter_statistics():
    """Retrieve the most recent parameter statistics from database."""
    try:
        with engine.connect() as conn:
            df = pd.read_sql(text("""
                SELECT Parameter, MonitoringCount, RecipeCount, AnalyzedAt
                FROM `model_schema`.`parameter_statistics`
                ORDER BY AnalyzedAt DESC, MonitoringCount DESC
            """), conn)
        
        if df.empty:
            print("No parameter statistics found in database.")
            return None
        
        print(f"✅ Parameter Statistics ({len(df)} unique parameters)")
        print(df.to_string(index=False))
        return df
    except Exception as e:
        print(f"Error retrieving parameter statistics: {str(e)}")
        return None

# Retrieve and display all stored analysis
print("\n" + "="*80)
print("STORED ANALYSIS RETRIEVAL")
print("="*80)

print("\n1. Latest Configuration Report:")
latest_report = get_latest_configuration_report()

print("\n" + "-"*80)
print("2. Coverage Analysis Results:")
coverage_results = get_latest_coverage_analysis()

print("\n" + "-"*80)
print("3. Parameter Statistics:")
param_stats = get_latest_parameter_statistics()

In [ ]:
def export_configuration_report_to_db(report_name='configuration_export'):
    """Export configuration report to model_schema.configuration_reports table."""
    if configs_df.empty:
        print("No configurations to export.")
        return None
    
    try:
        # Create export dataframe
        export_data = []
        for _, config in configs_df.iterrows():
            export_data.append({
                'ConfigurationId': int(config['ConfigurationId']),
                'ConfigurationName': str(config['ConfigurationName']),
                'MachineCode': str(config['MachineCode']),
                'MonitoringParametersCount': len(config['MonitoringParameters']),
                'RecipeParametersCount': len(config['RecipeParameters']),
                'MonitoringParameters': '; '.join(config['MonitoringParameters']),
                'RecipeParameters': '; '.join(config['RecipeParameters']),
                'IsActive': bool(config['IsActive']),
                'CreatedAt': str(config['CreatedAt']),
                'UpdatedAt': str(config['UpdatedAt']),
                'Description': str(config.get('Description', ''))
            })
        
        # Create summary statistics
        summary = {
            'total_configurations': len(configs_df),
            'active_configurations': int(len(configs_df[configs_df['IsActive'] == True])),
            'unique_machines': int(configs_df['MachineCode'].nunique()),
            'avg_monitoring_params': round(float(configs_df['n_monitoring_params'].mean()), 2),
            'avg_recipe_params': round(float(configs_df['n_recipe_params'].mean()), 2)
        }
        
        # Save to database
        with engine.connect() as conn:
            conn.execute(text("""
                INSERT INTO `model_schema`.`configuration_reports`
                (ReportName, ReportType, ReportData, Summary, ProcessedRows)
                VALUES (:name, :type, :data, :summary, :rows)
            """), {
                'name': report_name,
                'type': 'configuration_export',
                'data': json.dumps(export_data),
                'summary': json.dumps(summary),
                'rows': len(export_data)
            })
            conn.commit()
        
        print(f"✅ Configuration report '{report_name}' stored in model_schema.configuration_reports")
        print(f"   Total rows: {len(export_data)}")
        print(f"   Report Summary:")
        for key, value in summary.items():
            print(f"   - {key}: {value}")
        
        return pd.DataFrame(export_data)
    except Exception as e:
        print(f"❌ Error saving report to database: {str(e)}")
        return None

# Export the report to database
if not configs_df.empty:
    report_df = export_configuration_report_to_db('machine_configuration_export')
    if report_df is not None:
        print("\n✅ First few rows of the report:")
        print(report_df.head())
else:
    print("No configurations to export.")